# SecBERT Fine-Tuning for Cybersecurity NER

**Methodology**

```
jackaduma/SecBERT  (already pre-trained with MLM on cybersecurity corpora by its original authors)
        ↓
Fine-tune for Cybersecurity NER (Token Classification) on the CyberNER dataset
        ↓
Evaluate (Precision, Recall, F1 — seqeval, entity-level)
        ↓
Demo (Flask + React app in this repository)
```

- **Base model:** [`jackaduma/SecBERT`](https://huggingface.co/jackaduma/SecBERT) — a BERT model pre-trained by its original authors using Masked Language Modeling (MLM) on security-domain corpora (APTnotes, Stucco-Data, CASIE, SemEval-2018 Task 8). **We do not perform any MLM pre-training ourselves.**
- **Our contribution:** fine-tuning SecBERT for the cybersecurity Named Entity Recognition task (token classification, 31-label BIO schema with 15 cyber entity types) and evaluating it with standard NER metrics.

> **Before running:** switch to a GPU runtime — **Runtime → Change runtime type → T4 GPU** — then **Runtime → Run all**.

In [ ]:
%pip install -q -U transformers datasets evaluate seqeval accelerate

import torch
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠️ No GPU detected — go to Runtime → Change runtime type → T4 GPU, then restart.")

## 1. Get the dataset files

Two files from the repository are needed:

| File | Repo location |
|---|---|
| `cyberner_clean.csv` | `datasets/cyber/cyberner_clean.csv` |
| `ner_cyber_labels.json` | `config/ner_cyber_labels.json` |

**Option A (default):** the cell below opens a file picker — select both files from your machine.

**Option B:** put them in the top level of your Google Drive and uncomment the Drive block instead.

In [ ]:
import os

REQUIRED_FILES = ["cyberner_clean.csv", "ner_cyber_labels.json"]

# --- Option B: load from Google Drive (uncomment to use) ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/cyberner_clean.csv" "/content/drive/MyDrive/ner_cyber_labels.json" .

# --- Option A: upload from your machine (default) ---
missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
if missing:
    print(f"Please upload: {missing}")
    from google.colab import files
    files.upload()

missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
assert not missing, f"Still missing: {missing} — re-run this cell and select both files."
print("✓ Dataset files ready")

## 2. Load the CyberNER dataset and map labels

The CSV is token-per-row (`Word`, `Tag`, `Sentence_ID`). Raw tags are collapsed to the 31-label cyber-only schema via the `tag_mapping` in `ner_cyber_labels.json`; unmapped tags become `O`. This mirrors `scripts/evaluate_ner.py` in the repo exactly, so local re-evaluation reproduces these results.

In [ ]:
import json
import pandas as pd

with open("ner_cyber_labels.json", encoding="utf-8") as f:
    schema = json.load(f)

label_list = schema["label_list"]      # 31 labels: O + B-/I- of 15 cyber entity types
tag_mapping = schema["tag_mapping"]    # collapses raw dataset tags; unmapped -> O
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print(f"Labels ({len(label_list)}):", label_list)

df = pd.read_csv("cyberner_clean.csv")
df["Word"] = df["Word"].fillna("#")
df["Tag"] = df["Tag"].fillna("O").map(lambda t: tag_mapping.get(t, "O"))

grouped = df.groupby("Sentence_ID").agg({"Word": list, "Tag": list}).reset_index()
grouped["ner_tags"] = grouped["Tag"].apply(lambda tags: [label2id.get(t, label2id["O"]) for t in tags])
grouped = grouped.rename(columns={"Word": "tokens"})
print(f"Sentences: {len(grouped)}")

## 3. Train / validation / test splits

Same seeds as the repo's evaluation script: first 80/20 gives the held-out **test** set (`scripts/evaluate_ner.py` re-derives this identical split locally with `seed=42`), then 80/20 of the remainder gives train/validation.

In [ ]:
from datasets import Dataset

full_ds = Dataset.from_pandas(grouped[["tokens", "ner_tags"]])
split1 = full_ds.train_test_split(test_size=0.2, seed=42)            # split1["test"] = held-out test set
split2 = split1["train"].train_test_split(test_size=0.2, seed=42)
train_ds, val_ds, test_ds = split2["train"], split2["test"], split1["test"]
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

## 4. Tokenize with the SecBERT tokenizer and align labels

SecBERT ships its own security-domain WordPiece vocabulary. Words are tokenized with `is_split_into_words=True`; only the **first sub-token** of each word carries the word's label, remaining sub-tokens and special tokens are masked with `-100` so they are ignored by the loss and the metrics.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "jackaduma/SecBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.is_fast, "A fast tokenizer is required for word_ids()"

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)               # special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])    # first sub-token gets the word label
            else:
                label_ids.append(-100)               # remaining sub-tokens masked
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = train_ds.map(tokenize_and_align_labels, batched=True)
tokenized_val = val_ds.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_ds.map(tokenize_and_align_labels, batched=True)

## 5. Load SecBERT with a token-classification head

> **Expected warning:** SecBERT is published as a masked-language-model checkpoint, so loading it for token classification prints *"Some weights ... were newly initialized: ['classifier.weight', 'classifier.bias']"*. That is normal — the pre-trained encoder is reused and the new classification head is exactly what the fine-tuning below trains.

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

## 6. Evaluation metrics (seqeval — entity-level Precision / Recall / F1)

In [ ]:
import numpy as np
import evaluate

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_predictions = [
        [id2label[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## 7. Fine-tune (token classification)

In [ ]:
import inspect
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="secbert_ner",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    fp16=True,
    report_to="none",
)

# transformers renamed Trainer's `tokenizer` arg to `processing_class`; support both
trainer_kwargs = {}
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    **trainer_kwargs,
)

trainer.train()

## 8. Final evaluation on the held-out test set

Overall Precision / Recall / F1, plus a strict IOB2 per-entity classification report (same format as `scripts/evaluate_ner.py`).

In [ ]:
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

test_output = trainer.predict(tokenized_test)

print("Test-set entity-level metrics:")
for key in ("test_precision", "test_recall", "test_f1", "test_accuracy"):
    print(f"  {key.replace('test_', '').capitalize():<10} {test_output.metrics[key]:.4f}")

predictions = np.argmax(test_output.predictions, axis=-1)
labels = test_output.label_ids
true_predictions = [
    [id2label[p] for p, l in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]
true_labels = [
    [id2label[l] for p, l in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

print("\nPer-entity classification report (strict IOB2):")
print(classification_report(true_labels, true_predictions, mode="strict", scheme=IOB2, zero_division=0))

## 9. Save and export the fine-tuned model

In [ ]:
trainer.save_model("secbert_ner_final")
tokenizer.save_pretrained("secbert_ner_final")

!zip -r -q secbert_ner_final.zip secbert_ner_final
from google.colab import files
files.download("secbert_ner_final.zip")

# Alternatively, copy to Drive instead of downloading:
# !cp secbert_ner_final.zip /content/drive/MyDrive/

## 10. Deploy locally

1. Extract `secbert_ner_final.zip` so the model lands at **`models/secbert_ner_final/`** in the project root (it must contain `config.json`, `model.safetensors`, `tokenizer_config.json`, and the vocab/tokenizer files).
2. Reproduce the test-set evaluation locally: `python scripts/evaluate_ner.py` (writes `evaluation_results.json`).
3. Start the demo: `python backend/ner_api.py`, then `cd frontend && npm start`.